# 🔍 BLAST: Sequence Similarity Search

---

## Learning Objectives

- Understand how BLAST works
- Choose appropriate BLAST program
- Interpret BLAST results (E-value, identity)
- Run BLAST programmatically

---

## BLAST Algorithm Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         HOW BLAST WORKS                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  Step 1: SEEDING                                                        │
│  ─────────────────                                                      │
│  Query: ...VHLTPEEKSAV...                                              │
│              |||                                                        │
│  Database has index of all words of length W (default: 3 for protein)  │
│  Find all "seeds" - exact matches of short words                       │
│                                                                         │
│  Step 2: EXTENSION                                                      │
│  ──────────────────                                                     │
│  Query:  ...VHLTPEEKSAV...                                             │
│             ─────────                                                   │
│  Subject: ...VHLTPVEKSAV...                                            │
│                                                                         │
│  Extend seed in both directions while score increases                   │
│  Stop when score drops below threshold                                  │
│                                                                         │
│  Step 3: EVALUATION                                                     │
│  ───────────────────                                                    │
│  Calculate alignment statistics:                                        │
│  • Score (bit score)                                                   │
│  • E-value (expected number of random hits)                            │
│  • Identity percentage                                                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## BLAST Programs

```
┌──────────────┬──────────────┬──────────────┬──────────────────────────┐
│ Program      │ Query        │ Database     │ Use Case                 │
├──────────────┼──────────────┼──────────────┼──────────────────────────┤
│ BLASTN       │ Nucleotide   │ Nucleotide   │ DNA/RNA similarity       │
│ BLASTP       │ Protein      │ Protein      │ Protein similarity       │
│ BLASTX       │ Nucleotide   │ Protein      │ Translate DNA, search    │
│ TBLASTN      │ Protein      │ Nucleotide   │ Search translated DB     │
│ TBLASTX      │ Nucleotide   │ Nucleotide   │ Both translated          │
├──────────────┼──────────────┼──────────────┼──────────────────────────┤
│ megaBLAST    │ Nucleotide   │ Nucleotide   │ Highly similar seqs      │
│ PSI-BLAST    │ Protein      │ Protein      │ Distant homologs         │
│ PHI-BLAST    │ Protein      │ Protein      │ Pattern-based search     │
└──────────────┴──────────────┴──────────────┴──────────────────────────┘

    Query Type?         Database Type?         Use:
    ┌─────────┐         ┌─────────────┐
    │ DNA/RNA │────────►│  DNA/RNA    │───────► BLASTN
    └─────────┘         └─────────────┘
         │                    │
         │                    └────────────────► BLASTX (translate query)
         │
    ┌─────────┐         ┌─────────────┐
    │ Protein │────────►│  Protein    │───────► BLASTP
    └─────────┘         └─────────────┘
         │                    │
         │              ┌─────────────┐
         └─────────────►│  DNA/RNA    │───────► TBLASTN
                        └─────────────┘
```

---

## Understanding E-value

The **E-value** (Expect value) tells you how many hits with this score you would expect by chance.

```
    E-value interpretation:
    
    ┌──────────────┬────────────────────────────────────────────┐
    │ E-value      │ Interpretation                             │
    ├──────────────┼────────────────────────────────────────────┤
    │ < 10⁻⁵⁰      │ Essentially identical                      │
    │ 10⁻⁵⁰ - 10⁻¹⁰│ Clear homolog (same family)               │
    │ 10⁻¹⁰ - 10⁻⁵ │ Probable homolog (distant)                │
    │ 10⁻⁵ - 10⁻¹  │ Possible homolog (check carefully)        │
    │ > 10⁻¹       │ Likely random match                       │
    └──────────────┴────────────────────────────────────────────┘
    
    Lower E-value = Better match!
    
    E = K × m × n × e^(-λS)
    
    Where:
    m = query length
    n = database size
    S = raw score
    K, λ = statistical parameters
```

In [ ]:
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import SeqIO

def run_blast_online(sequence, program="blastp", database="nr", e_value=0.001):
    """
    Run BLAST search against NCBI.
    
    Parameters:
        sequence: Query sequence string
        program: BLAST program (blastn, blastp, blastx, etc.)
        database: Target database (nr, nt, swissprot, etc.)
        e_value: E-value threshold
        
    Returns:
        BLAST record object
    """
    print(f"Running {program} against {database}...")
    result_handle = NCBIWWW.qblast(
        program,
        database,
        sequence,
        expect=e_value
    )
    
    blast_record = NCBIXML.read(result_handle)
    return blast_record

def parse_blast_results(blast_record, max_hits=10):
    """
    Parse and display BLAST results.
    """
    print(f"\nFound {len(blast_record.alignments)} hits\n")
    print(f"{'Hit':<50} {'E-value':>12} {'Identity':>10}")
    print("-" * 75)
    
    for i, alignment in enumerate(blast_record.alignments[:max_hits]):
        for hsp in alignment.hsps:
            identity = hsp.identities / hsp.align_length * 100
            title = alignment.title[:47] + "..." if len(alignment.title) > 50 else alignment.title
            print(f"{title:<50} {hsp.expect:>12.2e} {identity:>9.1f}%")

print("BLAST functions defined.")
print("\nExample usage:")
print('  record = run_blast_online("MVHLTPEEKSAVTALWGKV...", "blastp", "swissprot")')
print('  parse_blast_results(record)')

---

## BLAST Output Format

```
Query= my_sequence

                                                                    Score     E
Sequences producing significant alignments:                         (Bits)  Value

sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens         289    2e-99
sp|P01942|HBA_MOUSE Hemoglobin subunit alpha OS=Mus musculus         278    5e-95
...

>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha
 Length=142

 Score = 289 bits (740),  Expect = 2e-99
 Identities = 141/142 (99%), Positives = 141/142 (99%), Gaps = 0/142 (0%)

Query  1    MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH-----   55
            MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH
Sbjct  1    MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH-----   55
```

In [ ]:
def format_blast_alignment(hsp, query_name="Query", subject_name="Sbjct", width=60):
    """
    Format a single BLAST alignment for display.
    """
    lines = []
    
    q_start = hsp.query_start
    s_start = hsp.sbjct_start
    
    for i in range(0, len(hsp.query), width):
        q_seq = hsp.query[i:i+width]
        match = hsp.match[i:i+width]
        s_seq = hsp.sbjct[i:i+width]
        
        # Calculate positions
        q_end = q_start + len(q_seq.replace('-', '')) - 1
        s_end = s_start + len(s_seq.replace('-', '')) - 1
        
        lines.append(f"{query_name:<6} {q_start:>5}  {q_seq}  {q_end:>5}")
        lines.append(f"       {' '*5}  {match}")
        lines.append(f"{subject_name:<6} {s_start:>5}  {s_seq}  {s_end:>5}")
        lines.append("")
        
        q_start = q_end + 1
        s_start = s_end + 1
    
    return '\n'.join(lines)

# Example alignment
print("Example BLAST alignment format:")
print()
print("Query     1  MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH  50")
print("             MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH")
print("Sbjct     1  MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH  50")

---

## 🏋️ Exercises

### Exercise 1: Find Homologs
Use BLASTP to find homologs of your favorite protein.

### Exercise 2: Compare Databases
Run the same search against nr vs swissprot and compare results.

### Exercise 3: Batch BLAST
Write a script to run BLAST for multiple sequences and save results.

---

## 📚 Resources

- [NCBI BLAST](https://blast.ncbi.nlm.nih.gov/)
- [BLAST Help](https://blast.ncbi.nlm.nih.gov/doc/blast-help/)
- [BioPython BLAST](https://biopython.org/docs/latest/api/Bio.Blast.html)